# Previsão do Tempo com OpenWeatherMap

Este notebook consulta a API do OpenWeatherMap e exibe informações meteorológicas completas com saída em áudio.

**Como usar:**
1. Gere sua chave de API gratuita em [openweathermap.org](https://openweathermap.org)
2. Substitua `SUA_API_KEY` pela sua chave
3. Execute as células

In [ ]:
# ============================================================
# PASSO 1: Importar as bibliotecas necessárias
# ============================================================

# 'requests' é a biblioteca que permite ao Python se comunicar
# com a internet. Usamos ela para chamar a API do OpenWeatherMap
# e obter os dados meteorológicos em tempo real.
import requests

# 'pyttsx3' é a biblioteca de texto para voz (Text-to-Speech).
# Ela usa o motor de voz já instalado no sistema operacional,
# funcionando sem internet, e faz o computador FALAR um texto.
import pyttsx3

# ============================================================
# PASSO 2: Configurar as constantes da aplicação
# ============================================================

# API_KEY é a sua chave de acesso ao serviço OpenWeatherMap.
# Funciona como uma 'senha' que identifica quem está fazendo
# a requisição. Gere a sua gratuitamente em:
# https://openweathermap.org/api
API_KEY = "SUA_API_KEY"  # Gere em: https://openweathermap.org/api

# BASE_URL é o endereço base da API que usamos.
# O endpoint '/weather' retorna dados do tempo ATUAL de uma cidade.
# (Para previsões de vários dias, o endpoint seria '/forecast'.)
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"


In [ ]:
# ============================================================
# DEFINIÇÃO DAS FUNÇÕES
# ============================================================
# Em Python, 'def' cria uma função — um bloco de código reutilizável
# que só executa quando é chamado. Funções recebem 'parâmetros'
# (valores de entrada) e podem retornar um resultado.
# Organizar o código em funções torna tudo mais legível e fácil
# de manter.


def obter_previsao(cidade, api_key):
    """Consulta a API e retorna os dados meteorológicos da cidade."""
    # --------------------------------------------------------
    # Montamos o dicionário de parâmetros da requisição.
    # A biblioteca 'requests' vai converter isso automaticamente
    # para o formato de URL: ?q=cidade&appid=chave&units=metric...
    # --------------------------------------------------------
    params = {
        "q": cidade,       # Nome da cidade que queremos consultar
        "appid": api_key,  # Nossa chave de autenticação na API
        "units": "metric", # Retorna temperaturas em Celsius
                           # (sem isso, viria em Kelvin por padrão)
        "lang": "pt_br"    # Retorna descrições em português do Brasil
                           # (ex: 'céu limpo' em vez de 'clear sky')
    }

    # requests.get() faz a requisição HTTP GET para a API.
    # O objeto 'response' contém o código de status (200, 404, etc.)
    # e os dados retornados pela API no formato JSON.
    response = requests.get(BASE_URL, params=params)

    # Retornamos o objeto response inteiro para que o código
    # que chamou esta função possa verificar o status e os dados.
    return response


def exibir_previsao(cidade, dados):
    """Formata e imprime as informações meteorológicas."""
    # --------------------------------------------------------
    # Extraindo dados do dicionário retornado pela API
    # --------------------------------------------------------
    # A API retorna um JSON que, ao ser convertido, vira um dicionário
    # Python. Aqui navegamos por esse dicionário usando chaves.
    # dados["main"] contém temperatura, umidade, pressão etc.
    main = dados["main"]

    # dados["wind"] contém informações sobre o vento (velocidade, direção).
    vento = dados["wind"]

    # dados["weather"] é uma LISTA de condições climáticas.
    # Geralmente tem só um item, por isso pegamos o índice [0].
    # ["description"] é o texto da condição (ex: 'céu limpo').
    # .capitalize() coloca a primeira letra em maiúscula.
    descricao = dados["weather"][0]["description"].capitalize()

    # --------------------------------------------------------
    # Extraindo cada valor numérico do dicionário 'main'
    # --------------------------------------------------------
    temp     = main["temp"]       # Temperatura atual em °C
    sensacao = main["feels_like"] # Sensação térmica em °C
    temp_min = main["temp_min"]   # Temperatura mínima do dia em °C
    temp_max = main["temp_max"]   # Temperatura máxima do dia em °C
    umidade  = main["humidity"]   # Umidade relativa do ar em %

    # vel_vento vem de dados["wind"]["speed"] em metros por segundo.
    vel_vento = vento["speed"]

    # --------------------------------------------------------
    # Formatando e imprimindo o relatório visual no console
    # --------------------------------------------------------
    # '='*45 repete o caractere '=' 45 vezes, criando uma linha
    # separadora visual. Ex: '=====================...'
    print(f"\n{'='*45}")

    # .title() coloca a primeira letra de cada palavra em maiúscula.
    # Ex: 'são paulo' → 'São Paulo'
    print(f"  Previsão do Tempo — {cidade.title()}")
    print(f"{'='*45}")
    print(f"  Condição      : {descricao}")

    # ':.1f' dentro da f-string formata o número float com
    # exatamente 1 casa decimal. Ex: 22.567 → '22.6'
    print(f"  Temperatura   : {temp:.1f}°C")
    print(f"  Sensação term.: {sensacao:.1f}°C")
    print(f"  Mín / Máx     : {temp_min:.1f}°C / {temp_max:.1f}°C")
    print(f"  Umidade       : {umidade}%")
    print(f"  Vento         : {vel_vento} m/s")
    print(f"{'='*45}\n")

    # Retornamos a temperatura e a descrição para que o código principal
    # possa usá-los na mensagem de áudio, sem precisar recalcular.
    return temp, descricao


def falar(mensagem):
    """Lê a mensagem em voz alta usando pyttsx3."""
    # pyttsx3.init() inicializa (liga) o motor de síntese de voz.
    # Ele detecta automaticamente o motor de voz do sistema operacional
    # (SAPI5 no Windows, espeak no Linux, NSSpeechSynthesizer no Mac).
    engine = pyttsx3.init()

    # setProperty() configura propriedades do motor de voz.
    # 'rate' é a velocidade da fala em palavras por minuto.
    # 150 é um valor confortável para entendimento — o padrão é ~200.
    engine.setProperty("rate", 150)

    # engine.say() coloca o texto na fila para ser falado.
    # Este comando NÃO inicia o áudio ainda — só agenda a fala.
    engine.say(mensagem)

    # engine.runAndWait() processa a fila de áudio e BLOQUEIA
    # o programa até o texto ser falado completamente.
    # Sem ele, o programa encerraria antes do áudio terminar.
    engine.runAndWait()


In [ ]:
# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================
# Aqui chamamos as funções que definimos acima, na ordem certa,
# para que tudo funcione em conjunto.

# --------------------------------------------------------
# PASSO 1: Solicitar o nome da cidade ao usuário
# --------------------------------------------------------
# input() exibe a mensagem e aguarda o usuário digitar algo.
# .strip() remove espaços extras no início e no fim do texto digitado.
# Ex: '  São Paulo  ' → 'São Paulo'
cidade = input("Digite a cidade: ").strip()

# --------------------------------------------------------
# PASSO 2: Consultar a API com a cidade informada
# --------------------------------------------------------
# Chamamos a função obter_previsao() que definimos acima.
# Ela faz a requisição e retorna o objeto 'response' com
# o código de status e os dados da API.
response = obter_previsao(cidade, API_KEY)

# ============================================================
# PASSO 3: Tratar a resposta com base no código de status HTTP
# ============================================================
# Usamos if/elif/else para tratar diferentes situações.
# Cada código HTTP indica um resultado diferente:
#   200 → Sucesso
#   404 → Recurso não encontrado (cidade inválida)
#   401 → Não autorizado (API key inválida)
#   outros → Algum erro inesperado

if response.status_code == 200:
    # --------------------------------------------------------
    # CASO: Requisição bem-sucedida
    # --------------------------------------------------------
    # response.json() converte o corpo da resposta (texto JSON)
    # em um dicionário Python de forma prática.
    # É equivalente a json.loads(response.text).
    dados = response.json()

    # Chamamos exibir_previsao() para mostrar os dados no console.
    # A função retorna a temperatura e a descrição para usarmos no áudio.
    temp, descricao = exibir_previsao(cidade, dados)

    # Montamos a frase que o computador vai FALAR.
    # ':.0f' formata o número sem casas decimais. Ex: 22.7 → '23'
    # Usamos parênteses para quebrar a string longa em duas linhas.
    mensagem_audio = (
        f"Em {cidade}, o tempo está {descricao}. "
        f"A temperatura é de {temp:.0f} graus Celsius."
    )

    # Chamamos a função falar() para reproduzir o áudio.
    falar(mensagem_audio)

elif response.status_code == 404:
    # --------------------------------------------------------
    # CASO: Cidade não encontrada
    # --------------------------------------------------------
    # O código 404 significa que a API não reconheceu o nome da cidade.
    # Informamos o usuário na tela E em áudio.
    msg = f"Cidade '{cidade}' não encontrada. Verifique o nome e tente novamente."
    print(f"Erro: {msg}")
    falar(msg)

elif response.status_code == 401:
    # --------------------------------------------------------
    # CASO: API key inválida ou ausente
    # --------------------------------------------------------
    # O código 401 significa que a chave de API não foi aceita.
    # Verifique se você substituiu 'SUA_API_KEY' pela chave real.
    msg = "API key inválida. Verifique sua chave em openweathermap.org."
    print(f"Erro: {msg}")
    falar(msg)

else:
    # --------------------------------------------------------
    # CASO: Qualquer outro erro inesperado
    # --------------------------------------------------------
    # Para erros que não conhecemos previamente, exibimos o código
    # de status para que o usuário possa investigar.
    msg = f"Erro ao consultar a API. Código HTTP: {response.status_code}."
    print(f"Erro: {msg}")
    falar(msg)
